<a href="https://colab.research.google.com/github/eaedk/Machine-Learning-Tutorials-Updated/blob/main/01_tabular/03_regression_quest_california.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏠 Quest · Tabular regression: The price of a Californian dream

**1990, United States census.** For every *block group* of California (a neighbourhood of ~1 500 people), the census
records the median income, the age of the houses, the number of rooms, the population… and the **median house value**.
In 1997, statisticians R. Kelley Pace and Ronald Barry turned it into a dataset that every ML learner has met since.

**Your mission:** predict the median house value of a neighbourhood, a **regression** task (we predict a *number*,
not a class).

| You will practise | Concept |
|---|---|
| Regression metrics | MAE, RMSE, R², and when to use which |
| A map from two columns | geographic EDA |
| Linear regression and its residuals | what a model gets wrong, and where |
| Feature engineering | ratios, distances, log transforms |
| Trees vs linear models | non-linearity, interactions |
| Honest evaluation | K-fold CV, holdout test |

🧩 Complete the `# TODO` cells. `check(...)` tells you if you're on track. 💡 Solutions are hidden below each exercise.

In [ ]:
# ⚙️ Setup: run me first! Finds the course toolkit `mlquest` (Windows, macOS, Linux and Colab).
import os, sys, subprocess, pathlib
if "google.colab" in sys.modules:  # on Colab: grab the repo (once) and the few missing packages
    if not pathlib.Path("/content/Machine-Learning-Tutorials-Updated").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/eaedk/Machine-Learning-Tutorials-Updated", "/content/Machine-Learning-Tutorials-Updated"], check=True)
    os.chdir("/content/Machine-Learning-Tutorials-Updated")
ROOT = pathlib.Path.cwd()
while not (ROOT / "mlquest").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from mlquest import *  # paths, seed_everything, get_device, quest/checkpoint, Leaderboard...
set_style()
SEED = seed_everything(42)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing

quest("The price of a Californian dream", chapter="Track 1 · Tabular · Regression", xp=300)

## 1. Load the data 📥

The target `MedHouseVal` is expressed in **units of $100 000** (so `2.5` means $250 000).

In [ ]:
data = fetch_california_housing(as_frame=True, data_home=str(DATA_DIR / "sklearn"))
df = data.frame
print(data.DESCR[:1200])

In [ ]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=0.2, random_state=SEED)
y, y_test = train.pop("MedHouseVal"), test.pop("MedHouseVal")
print(train.shape, test.shape)
train.head()

## 2. How do we score a regression? 📏

Three metrics you will meet everywhere:

| Metric | Formula (idea) | Reads as | Sensitive to big errors? |
|---|---|---|---|
| **MAE** | mean of \|error\| | "on average I'm off by…" | no |
| **RMSE** | √(mean of error²) | same unit as the target | **yes**, squares them |
| **R²** | 1 − error² / variance | share of variance explained (1 = perfect, 0 = predicts the mean) | yes |

### 🎯 Exercise 1: implement MAE and RMSE yourself
Using numpy only (`np.abs`, `np.mean`, `np.sqrt`).

In [ ]:
def mae(y_true, y_pred):
    return ...   # TODO

def rmse(y_true, y_pred):
    return ...   # TODO

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
def mae(y_true, y_pred):
    return np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))
```

</details>

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
a, b = np.array([1.0, 2.0, 3.0, 10.0]), np.array([1.5, 2.0, 2.0, 6.0])
check(np.isclose(mae(a, b), mean_absolute_error(a, b)) and np.isclose(rmse(a, b), np.sqrt(mean_squared_error(a, b))),
      f"Both correct! On this example MAE = {mae(a, b):.2f} but RMSE = {rmse(a, b):.2f}: the one big error (10 vs 6) weighs a lot more in RMSE.",
      "MAE = mean(|y_true - y_pred|), RMSE = sqrt(mean((y_true - y_pred)**2)).")

### 🎯 Exercise 2: the "no-brain" baseline
Predict the **mean** price of the training set for every neighbourhood. What is its RMSE on the training targets?

In [ ]:
baseline_pred = ...   # TODO: an array full of y.mean(), same length as y
baseline_rmse = rmse(y, baseline_pred)
print(f"Baseline RMSE: {baseline_rmse:.3f}  (= ${baseline_rmse * 100_000:,.0f})")

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
baseline_pred = np.full(len(y), y.mean())
baseline_rmse = rmse(y, baseline_pred)
print(f"Baseline RMSE: {baseline_rmse:.3f}  (= ${baseline_rmse * 100_000:,.0f})")
```

</details>

In [ ]:
check(abs(baseline_rmse - y.std(ddof=0)) < 1e-6,
      "Yes! The RMSE of 'predict the mean' is exactly the standard deviation of the target. Every model must beat this.",
      "baseline_pred should be np.full(len(y), y.mean()).")
checkpoint("california/metrics", xp=40)

## 3. EDA: a map made of numbers 🗺️

Plot each neighbourhood at its (longitude, latitude), coloured by price and sized by population. Look: California
appears, with the San Francisco Bay Area and Los Angeles glowing along the coast.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(train["Longitude"], train["Latitude"], c=y, s=train["Population"] / 300,
                cmap="magma", alpha=0.4)
plt.colorbar(sc, label="median house value ($100k)")
ax.set(xlabel="longitude", ylabel="latitude", title="California, drawn by house prices")
ax.annotate("San Francisco", (-122.42, 37.77), xytext=(-125, 36.5), arrowprops=dict(arrowstyle="->"))
ax.annotate("Los Angeles", (-118.24, 34.05), xytext=(-121, 33), arrowprops=dict(arrowstyle="->"))
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(y, bins=60, ax=axes[0]).set(title="Target distribution")
sns.scatterplot(x=train["MedInc"], y=y, alpha=0.1, ax=axes[1]).set(title="Median income vs house value")
plt.tight_layout(); plt.show()

**Observations**
- 📍 **Location matters** (coast, big cities).
- 💵 **Median income** is strongly correlated with price.
- 🧱 A strange wall at **5.0**: values were **capped** at $500 001 in the census. The model will never see higher
  prices, and those rows are hard to predict. (In a real project you'd discuss this with the data owner.)

### 🎯 Exercise 3: which feature correlates most with the target?

In [ ]:
correlations = ...    # TODO: Pearson correlation of every column of train with y (hint: train.corrwith(y))
best_feature = ...    # TODO: name of the feature with the highest absolute correlation
correlations.sort_values()

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
correlations = train.corrwith(y)
best_feature = correlations.abs().idxmax()
correlations.sort_values()
```

</details>

In [ ]:
check(best_feature == "MedInc", "MedInc it is (~0.69): richer neighbourhoods, pricier houses.",
      "Use correlations.abs().idxmax().")

## 4. A first model: linear regression 📈

Linear regression (Legendre 1805, Gauss 1809!) fits `price ≈ w₁·MedInc + w₂·HouseAge + … + b`. It is fast,
interpretable, and a must-have baseline.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
lb = Leaderboard("california", metric="RMSE", higher_is_better=False)
lb.add("Predict the mean", baseline_rmse)

def cv_rmse(model, X_):
    scores = -cross_val_score(model, X_, y, cv=cv, scoring="neg_root_mean_squared_error")
    return scores.mean(), scores.std()

linreg = make_pipeline(StandardScaler(), LinearRegression())
score, std = cv_rmse(linreg, train)
lb.add("Linear regression", score, std=round(std, 4))

### Where is the model wrong? Residuals 🔬
A **residual** = true value − prediction. Plotting residuals is the best debugging tool for regression.

In [ ]:
pred = cross_val_predict(linreg, train, y, cv=cv)
residuals = y - pred
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(pred, y, alpha=0.1); axes[0].plot([0, 5], [0, 5], "r--")
axes[0].set(xlabel="prediction", ylabel="truth", title="Truth vs prediction")
sc = axes[1].scatter(train["Longitude"], train["Latitude"], c=residuals, cmap="coolwarm", vmin=-2, vmax=2, s=3)
plt.colorbar(sc, label="residual (truth − prediction)")
axes[1].set(title="Residuals on the map")
plt.tight_layout(); plt.show()

The map of residuals is not random: the model **systematically underestimates** the coast (red) and overestimates
inland areas (blue). A linear function of longitude/latitude can't draw a coastline! That's a hint for features…

## 5. 🛠️ Feature engineering

### 🎯 Exercise 4: create better features
Complete `make_features` with:
- `rooms_per_person` = `AveRooms` / `AveOccup`
- `bedroom_ratio` = `AveBedrms` / `AveRooms`
- `log_population` = log(1 + `Population`) (`np.log1p`): squashes the huge values
- `dist_sf` and `dist_la`: distance (in degrees, Euclidean is fine) to San Francisco (-122.42, 37.77) and Los Angeles (-118.24, 34.05)

In [ ]:
def make_features(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    data["rooms_per_person"] = ...
    data["bedroom_ratio"] = ...
    data["log_population"] = ...
    data["dist_sf"] = ...    # hint: np.sqrt((data["Longitude"] + 122.42) ** 2 + (data["Latitude"] - 37.77) ** 2)
    data["dist_la"] = ...
    return data

X, X_test = make_features(train), make_features(test)
X.head()

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
def make_features(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    data["rooms_per_person"] = data["AveRooms"] / data["AveOccup"]
    data["bedroom_ratio"] = data["AveBedrms"] / data["AveRooms"]
    data["log_population"] = np.log1p(data["Population"])
    data["dist_sf"] = np.sqrt((data["Longitude"] + 122.42) ** 2 + (data["Latitude"] - 37.77) ** 2)
    data["dist_la"] = np.sqrt((data["Longitude"] + 118.24) ** 2 + (data["Latitude"] - 34.05) ** 2)
    return data

X, X_test = make_features(train), make_features(test)
X.head()
```

</details>

In [ ]:
check(X.shape[1] == 13 and X["dist_sf"].min() < 0.1 and X.isna().sum().sum() == 0,
      "5 new features, no missing values. Let's see if the linear model likes them.",
      "Expected 13 columns (8 original + 5 new) and dist_sf close to 0 near San Francisco.")
score, std = cv_rmse(make_pipeline(StandardScaler(), LinearRegression()), X)
lb.add("Linear regression + features", score, std=round(std, 4))

## 6. 🌳 Trees: let the model draw the coastline

Decision trees split the space with simple rules (`Latitude < 34.5 and MedInc > 5…`). Ensembles of trees capture
non-linearities and interactions automatically: perfect for maps.

### 🎯 Exercise 5: cross-validate a Random Forest and a Gradient Boosting model on `X`

In [ ]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, n_jobs=-1, random_state=SEED)
gb = HistGradientBoostingRegressor(max_iter=500, learning_rate=0.08, random_state=SEED)
for name, model in [("Random Forest", rf), ("Gradient Boosting", gb)]:
    score, std = ...     # TODO: use cv_rmse
    lb.add(name, score, std=round(std, 4))
lb.show()

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, n_jobs=-1, random_state=SEED)
gb = HistGradientBoostingRegressor(max_iter=500, learning_rate=0.08, random_state=SEED)
for name, model in [("Random Forest", rf), ("Gradient Boosting", gb)]:
    score, std = cv_rmse(model, X)
    lb.add(name, score, std=round(std, 4))
lb.show()
```

</details>

In [ ]:
check(lb.best()["RMSE"] < 0.5,
      f"Huge jump: from {baseline_rmse:.2f} (mean) to {lb.best()['RMSE']:.2f}. Trees learnt the geography.",
      "The tree models should reach an RMSE below 0.5. Did you pass X (with features)?")
checkpoint("california/trees", xp=80)

## 7. 🎯 A trick for skewed targets: predict the log

Prices are right-skewed (many cheap, few very expensive). Training on `log(price)` often helps, and makes the model
care about *relative* errors (10% off) rather than absolute ones. `TransformedTargetRegressor` handles the
`log`/`exp` round trip for us.

### 🎯 Exercise 6: wrap the gradient boosting model with a log-transformed target

In [ ]:
from sklearn.compose import TransformedTargetRegressor

gb_log = ...   # TODO: TransformedTargetRegressor(regressor=<a new HistGradientBoostingRegressor>, func=np.log1p, inverse_func=np.expm1)
score, std = cv_rmse(gb_log, X)
lb.add("Gradient Boosting (log target)", score, std=round(std, 4))

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
from sklearn.compose import TransformedTargetRegressor

gb_log = TransformedTargetRegressor(
    regressor=HistGradientBoostingRegressor(max_iter=500, learning_rate=0.08, random_state=SEED),
    func=np.log1p, inverse_func=np.expm1,
)
score, std = cv_rmse(gb_log, X)
lb.add("Gradient Boosting (log target)", score, std=round(std, 4))
```

</details>

Did it help? Compare with the `std` before celebrating: on this dataset the effect is small. On others (house prices
in the next notebook, sales, durations), it is a game changer.

## 8. 🏁 Final test

In [ ]:
best_name = lb.best()["model"]
final_model = {"Gradient Boosting": gb, "Gradient Boosting (log target)": gb_log, "Random Forest": rf}.get(best_name, gb)
final_model.fit(X, y)
test_pred = final_model.predict(X_test)
print(f"Final model: {best_name}")
print(f"Test RMSE = {rmse(y_test, test_pred):.3f}  |  MAE = {mae(y_test, test_pred):.3f} "
      f"(≈ ${mae(y_test, test_pred) * 100_000:,.0f} off on average)  |  R² = {1 - np.sum((y_test - test_pred) ** 2) / np.sum((y_test - y_test.mean()) ** 2):.3f}")
make_submission(pd.DataFrame({"block_id": test.index, "MedHouseVal": test_pred.round(4)}), "california_submission.csv")

## 🏁 Quest complete!

You learnt the three regression metrics, read residuals on a map, engineered geographic features, and saw trees
beat linear models when the world isn't linear.

### 🚀 Your turn (bonus XP)
1. Add distances to San Diego (-117.16, 32.72) and Sacramento (-121.49, 38.58). Better?
2. Rotate the coordinates: add `Longitude + Latitude` and `Longitude - Latitude` (trees love axis-aligned splits).
3. Try `LGBMRegressor` from `lightgbm` and tune `num_leaves` and `learning_rate`.
4. Remove the capped rows (`y >= 5`) from training only. What happens to the test score? Why is it dangerous?

Next: `04_regression_app_house_prices.ipynb`, the Kaggle classic with 79 features.

In [ ]:
checkpoint("california/complete", xp=180)
badge("Californian Appraiser", "🏠")
progress()